# Lab 13 — AlexNet: Filters, Feature Maps & Transfer Learning

*Open up a network that already works, look at what it learned, then retrain it on your own images.*

## Objectives

1. Load a pretrained AlexNet and inspect its layers and shapes
2. Visualize first-layer filters and intermediate feature maps
3. Preprocess arbitrary images into the tensor the network expects
4. Retrain the final layer on a dataset you assembled

## Interactive labs for this week

- [`alexnet.html`](../interactive/alexnet.html)

> Runs on CPU, but a **GPU runtime is much faster** for this one: *Runtime → Change runtime type → T4 GPU*

---

In [ ]:
# --- setup: run this cell first, every session ---------------------------------
import builtins as __builtin__
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.functional import *

import torchvision
from torchvision import datasets, models, transforms
from torchvision.models import *

from scipy import stats, signal, integrate
from sympy import Symbol

from skimage import io
from skimage import io as skio
from skimage.transform import resize
from skimage.util import montage
from imageio.v2 import imread

from IPython.display import display, HTML

# One device for the whole notebook. A GPU runtime is faster, but nothing here
# requires one -- switch with Runtime -> Change runtime type -> T4 GPU.
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def GPU(data):
    """A learnable float tensor on the active device."""
    return torch.tensor(data, requires_grad=True, dtype=torch.float, device=device)


def GPU_data(data):
    """A fixed (non-learnable) float tensor on the active device."""
    return torch.tensor(data, requires_grad=False, dtype=torch.float, device=device)


print('Running on:', device)

# This lab logs experiments to Weights & Biases.
!pip install -q wandb
import wandb as wb

#[Tensorspace](https://tensorspace.org/html/playground/alexnet.html)

![alt text](https://miro.medium.com/max/750/1*IlzW43-NtJrwqtt5Xy3ISA.jpeg)

![alt text](https://d3i71xaburhd42.cloudfront.net/38211dc39e41273c0007889202c69f841e02248a/2-Figure1-1.png)

![alt text](https://upload.wikimedia.org/wikipedia/commons/thumb/1/16/Alexnet.png/1280px-Alexnet.png)

![alt text](https://cv-tricks.com/wp-content/uploads/2017/03/alexnet_small-1.png)

![alt text](https://www.learnopencv.com/wp-content/uploads/2017/11/convolution-example-matrix.gif)

<img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/Max_pooling.png">

<img src=https://ujwlkarn.files.wordpress.com/2016/08/giphy.gif width="900">


In [ ]:
import matplotlib.pyplot as plt
import imageio
import torch
import torchvision
from torchvision import models, transforms
import numpy as np
from torchvision.models import *
from PIL import Image
import requests
from torchvision import models
from torchsummary import summary
from skimage.util import montage
from torch.autograd import Variable
from skimage import io as skio
from scipy import signal

In [ ]:
def scale(x):
    return (x - np.min(x))/np.max(x)
def scale1(x):
    return (x-np.min(x))/(np.max(x)-np.min(x))

In [ ]:
def plot(x):
    fig, ax = plt.subplots()
    im = ax.imshow(x,cmap='gray')
    ax.axis('off')
    fig.set_size_inches(10, 10)
    plt.show()

In [ ]:
image = skio.imread("http://ian-albert.com/games/super_mario_bros_maps/mario-2-2.gif")
image = image[0] if image.ndim == 4 else image  # animated GIF -> first frame
image = image[:,0:700,:]
plot(image)

In [ ]:
coin = image[185:200,224:239,:]

In [ ]:
plot(coin)

In [ ]:
image = image[60:,0:700,:]
plot(image)

In [ ]:
image=np.mean(image,axis=2)
coin=np.mean(coin,axis=2)
image=scale1(image)
coin=scale1(coin)

In [ ]:
plot(image)


In [ ]:
plot(coin)

In [ ]:
coin.shape

In [ ]:
image = image - np.mean(image)
coin = coin - np.mean(coin)

In [ ]:
z=signal.convolve2d(image, np.rot90(coin, 2))

In [ ]:
plot(z)

In [ ]:
plot(z==np.max(z))

In [ ]:
[y,x] = np.where(z == np.amax(z))

In [ ]:
plt.plot(x,-y,'r.',markersize=30)
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
fig, ax = plt.subplots()
im = ax.imshow(image, cmap = 'gray')
ax.axis('off')
fig.set_size_inches(20, 10)
ax.scatter(x-6, y-6, c='#f97306', s=80) #https://xkcd.com/color/rgb/
plt.show()

In [ ]:
net = alexnet(weights=AlexNet_Weights.DEFAULT).to(device)

In [ ]:
normalize = transforms.Normalize(
   mean=[0.485, 0.456, 0.406],
   std=[0.229, 0.224, 0.225]
)
preprocess = transforms.Compose([
   transforms.Resize(256),
   transforms.CenterCrop(224),
   transforms.ToTensor(),
   normalize
])

In [ ]:
im = imageio.imread('https://www.medicalnewstoday.com/content/images/articles/322/322868/golden-retriever-puppy.jpg')

In [ ]:
plot(im)

In [ ]:
image = Image.fromarray(im) #convert to pil

In [ ]:
img_tensor = preprocess(image)

In [ ]:
img_tensor = img_tensor.unsqueeze_(0)

In [ ]:
img_tensor.shape

In [ ]:
img_variable = img_tensor.to(device)

In [ ]:
img_variable.shape

In [ ]:
%%timeit
out = net(img_variable)

In [ ]:
out = net(img_variable)

In [ ]:
out.shape

In [ ]:
out

In [ ]:
label_index = out.cpu().data.numpy().argmax()

In [ ]:
label_index

In [ ]:
top_list = np.flip(np.argsort(out.cpu().data.numpy())[0][-10:])

In [ ]:
LABELS_URL = 'https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json'

In [ ]:
labels = {i: name for i, name in enumerate(requests.get(LABELS_URL).json())}

In [ ]:
print(labels[label_index])

In [ ]:
for i in range(10):
    print(labels[top_list[i]])

In [ ]:
net

In [ ]:
summary(net, (3, 224, 224))

In [ ]:
out = net.features[0](img_variable).cpu().detach().numpy()

In [ ]:
plot(out[0,0,:,:])

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:6](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
im = imageio.imread('http://bocasurfcam.com/most_recent_image.php')

In [ ]:
plot(im)

In [ ]:
def load_im(im):
    image = Image.fromarray(im) #convert to pil
    img_tensor = preprocess(image)
    img_tensor = img_tensor.unsqueeze_(0)
    img_variable = img_tensor.to(device)
    return img_variable

In [ ]:
out = net(load_im(im))

In [ ]:
def inference(im):
    out = net(load_im(im))
    label_index = out.cpu().data.numpy().argmax()
    top_list = np.flip(np.argsort(out.cpu().data.numpy())[0][-10:])
    print(labels[label_index])
    print('____')
    for i in range(10):
        print(labels[top_list[i]])

In [ ]:
inference(im)

In [ ]:
net

In [ ]:
w1 = net.features[0].weight.data.cpu().numpy()

In [ ]:
w1.shape

In [ ]:
w1 = np.transpose(w1,(0,2,3,1))

In [ ]:
w1.shape

In [ ]:
Wp = (w1 - np.mean(w1))/np.std(w1)

In [ ]:
M = montage(Wp, channel_axis=-1)

In [ ]:
M.shape

In [ ]:
plot(Wp[0,:,:,:])

In [ ]:
plot(np.random.random((11,11,3)))

In [ ]:
plot(montage(Wp, channel_axis=-1))

In [ ]:
plot(montage(w1, channel_axis=-1))

In [ ]:
plot(scale1(montage(np.transpose(net.features[0].weight.data.cpu().numpy(),(0,2,3,1)), channel_axis=-1)))

In [ ]:
plot(scale(montage(np.transpose(net.features[0].weight.data.cpu().numpy(),(0,2,3,1)), channel_axis=-1)))

In [ ]:
img_variable.flatten().shape

In [ ]:
out = net.features[0](img_variable).cpu().detach().numpy()

In [ ]:
out.shape

In [ ]:
plot(out[0,0,:,:])

In [ ]:
img_variable.shape

In [ ]:
3*224*224

In [ ]:
64*55*55

In [ ]:
out = np.squeeze(out)

In [ ]:
plot(montage(out))

In [ ]:
plot(scale(montage(np.transpose(net.features[0].weight.data.cpu().numpy(),(0,2,3,1)), channel_axis=-1)))

In [ ]:
plot(montage(np.squeeze(net.features[0:1](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:2](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:3](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:4](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:5](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:6](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:7](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:8](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:9](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:10](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:11](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.features[0:12](img_variable).cpu().detach().numpy())))

In [ ]:
plot(montage(np.squeeze(net.avgpool(net.features[0:13](img_variable)).cpu().detach().numpy())))

In [ ]:
net.features[0:13](img_variable).cpu().detach().numpy().shape

In [ ]:
net

In [ ]:
net.avgpool(net.features[0:13](img_variable)).shape

In [ ]:
net.avgpool(net.features[0:13](img_variable)).flatten().shape[0]

In [ ]:
plt.plot(np.arange(9216),net.avgpool(net.features[0:13](img_variable)).flatten().cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
9216 / (3*224*224) # percentage of numbers left, vs original pixels

In [ ]:
net.classifier[0](net.avgpool(net.features[0:13](img_variable)).flatten()).shape

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:2](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:3](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:5](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
plt.plot(np.arange(4096),net.classifier[0:6](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
plt.plot(np.arange(1000),net.classifier[0:7](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy())
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
out = net.classifier[0:7](net.avgpool(net.features[0:13](img_variable)).flatten()).cpu().detach().numpy()

In [ ]:
def softmax(x):

    s = np.exp(x)

    s = s / np.sum(s)

    return s

In [ ]:
out = softmax(out)

In [ ]:
plt.plot(np.arange(1000),out)
fig = plt.gcf()
fig.set_size_inches(10, 10)

In [ ]:
label_index = out.argmax()

In [ ]:
label_index

In [ ]:
top_list = np.flip(np.argsort(out))[0:10]

In [ ]:
top_list

In [ ]:
for i in range(10):
    print(labels[top_list[i]])

---

## Transfer learning: retrain AlexNet on your own images

Everything above inspects a network that already works. Now we point it at a problem
it has never seen.

We use the classic ants-vs-bees set — 245 training photos, 153 for validation. Small
enough to train on a CPU runtime, hard enough to be interesting.

Swap in your own folder and nothing else changes. **The folder names are the class
labels:**

```
mydata/
├── train/burnt/ ...   train/good/ ...
└── val/burnt/   ...   val/good/  ...
```

See [`../setup/drive.md`](../setup/drive.md) for getting your own data in from Drive.

In [ ]:
import urllib.request
import os, urllib.request, zipfile

if not os.path.isdir('hymenoptera_data'):
    urllib.request.urlretrieve(
        'https://download.pytorch.org/tutorial/hymenoptera_data.zip', 'hym.zip')
    zipfile.ZipFile('hym.zip').extractall('.')

DATA = 'hymenoptera_data'          # <- point this at your own folder in Drive

In [ ]:
from torch.utils.data import DataLoader

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),           # AlexNet's input size
    transforms.RandomHorizontalFlip(),       # augmentation: training only
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_set = datasets.ImageFolder(f'{DATA}/train', transform=train_tf)
val_set   = datasets.ImageFolder(f'{DATA}/val',   transform=val_tf)

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=16)

In [ ]:
# Always check this before training. If the classes are wrong, everything after is wrong.
print(train_set.classes)
print(train_set.class_to_idx)
print(len(train_set), 'train /', len(val_set), 'val')

In [ ]:
images, targets = next(iter(train_loader))
print(images.shape)                          # (batch, 3, 224, 224)
plot(montage(images[:9, 0, :, :].numpy()))   # `plot` is defined earlier in this lab

Those normalization numbers are the ImageNet statistics AlexNet was trained on. Use
them whenever you start from pretrained weights.

AlexNet ends in a 1000-way layer, one output per ImageNet class. Replace that last
layer with one that has an output per class of *yours*. Everything before it — the
filters and feature maps you just visualised — is kept.

In [ ]:
net = alexnet(weights=AlexNet_Weights.DEFAULT)
net.classifier[6] = nn.Linear(4096, len(train_set.classes))
net = net.to(device)
net.classifier

In [ ]:
# Freeze the learned features; train only the new head.
for p in net.features.parameters():
    p.requires_grad = False

optimizer = torch.optim.Adam(net.classifier.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss()

In [ ]:
for epoch in range(3):
    net.train()
    total = 0.0
    for x, t in train_loader:
        x, t = x.to(device), t.to(device)
        optimizer.zero_grad()
        loss = loss_fn(net(x), t)
        loss.backward()
        optimizer.step()
        total += loss.item()
    print(f'epoch {epoch}  loss {total/len(train_loader):.4f}')

In [ ]:
net.eval()
correct = n = 0
with torch.no_grad():
    for x, t in val_loader:
        x, t = x.to(device), t.to(device)
        correct += (net(x).argmax(1) == t).sum().item()
        n += t.shape[0]
print(f'validation accuracy: {correct}/{n} = {correct/n:.1%}')

Three epochs on a frozen backbone should land in the 90s. ImageNet contains neither
of these classes — what you retrained is one 4096-to-2 layer sitting on features that
already knew what edges and texture are.

Save the weights so a new session can pick them up (see `drive.md` §6).

In [ ]:
torch.save(net.state_dict(), 'alexnet_antsbees.pt')

# reload into a fresh model
net2 = alexnet()
net2.classifier[6] = nn.Linear(4096, len(train_set.classes))
net2.load_state_dict(torch.load('alexnet_antsbees.pt', weights_only=True))
net2.eval()
print('reloaded')

---

## Homework

1. Assemble a two-class image dataset of your own (30+ images per class) and retrain AlexNet on it.
2. Report accuracy and show the feature maps for one correct and one incorrect prediction.

Submit as a markdown report in your course GitHub repo, with plots committed alongside it.

<details><summary>Where this lab came from</summary>

- `_source/notebooks/MathData24/Feature_Maps__1GayNzpg.ipynb` cells 0–117 (116 kept)

</details>